Part 1: Heart Disease

In [ ]:
import pandas as pd

DATA_FILE = "cleveland.csv"

cleveland_data = pd.read_csv(DATA_FILE)

cleveland_data.head()

In [ ]:
columns = cleveland_data.columns.tolist()
columns.remove('num')  # Remove the "num" column for the features list to choose from for target variables
print(columns)

In [ ]:
# Give a heart disease true or false column based on the "num" column
cleveland_data['heart_disease'] = cleveland_data['num'] > 0
cleveland_data.head()

In [ ]:
# Generate a few different permutations of headings to test k-nearest neighbors on, and see which one performs the best.
from itertools import combinations
from math import comb
import random


def generate_subsets(colums, n, attr, seed=None):
    # Remove duplicate values from the input while preserving their order.
    values = list(dict.fromkeys(colums))

    if not 0 <= attr <= len(values):
        raise ValueError("k must be between 0 and the number of unique values")

    total = comb(len(values), attr)
    if not 0 <= n <= total:
        raise ValueError(f"n must be between 0 and {total}")

    rng = random.Random(seed)
    subsets = list(combinations(values, attr))
    yield from rng.sample(subsets, n)

In [ ]:
sets = list()
for subset in generate_subsets(columns, n=20, attr=4, seed=None):
    sets.append(subset)
    print(sets[-1])

In [ ]:
from collections.abc import Sequence
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Each process gets its own copy of these values through the initializer.
_WORKER_DATA = None
_WORKER_Y = None
_WORKER_SPLITS = None
_WORKER_NEIGHBOR_VALUES = None


def _predict_knn(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    n_neighbors: int,
) -> np.ndarray:
    """Predict binary labels using Euclidean-distance KNN."""
    distances = np.sum(
        (X_test[:, None, :] - X_train[None, :, :]) ** 2,
        axis=2,
    )
    nearest = np.argpartition(
        distances,
        kth=n_neighbors - 1,
        axis=1,
    )[:, :n_neighbors]

    predictions = []
    for neighbor_indices in nearest:
        votes = np.bincount(
            y_train[neighbor_indices],
            minlength=2,
        )
        # A tie is classified as 0 because np.argmax chooses the first max.
        predictions.append(int(np.argmax(votes)))

    return np.asarray(predictions)


def _init_knn_worker(data, y, split_indices, neighbor_values):
    """Initialize data shared by tasks running in each process."""
    global _WORKER_DATA
    global _WORKER_Y
    global _WORKER_SPLITS
    global _WORKER_NEIGHBOR_VALUES

    _WORKER_DATA = data
    _WORKER_Y = y
    _WORKER_SPLITS = split_indices
    _WORKER_NEIGHBOR_VALUES = neighbor_values


def _evaluate_subset_split(task):
    """Evaluate one feature subset on one train/test split."""
    subset, split_index = task
    split_seed, train_ids, test_ids = _WORKER_SPLITS[split_index]

    X = (
        _WORKER_DATA.loc[:, list(subset)]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    X_train = X[train_ids]
    X_test = X[test_ids]
    y_train = _WORKER_Y[train_ids]
    y_test = _WORKER_Y[test_ids]

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    rows = []

    for n_neighbors in _WORKER_NEIGHBOR_VALUES:
        predictions = _predict_knn(
            X_train,
            y_train,
            X_test,
            n_neighbors,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            predictions,
            labels=[0, 1],
        ).ravel()

        rows.append(
            {
                "attributes": subset,
                "n_neighbors": n_neighbors,
                "split_seed": split_seed,
                "precision": precision_score(
                    y_test,
                    predictions,
                    zero_division=0,
                ),
                "recall": recall_score(
                    y_test,
                    predictions,
                    zero_division=0,
                ),
                "f1": f1_score(
                    y_test,
                    predictions,
                    zero_division=0,
                ),
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
            }
        )

    return rows


def run_knn_on_subsets(
    data: pd.DataFrame,
    subsets: Sequence[tuple[str, ...]],
    target_column: str = "num",
    test_size: float = 0.2,
    seed: int | None = None,
    iterations: int = 10,
    neighbor_values: Sequence[int] = (1, 3, 5, 7, 9, 11, 13, 15),
    processes: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run repeated train/test evaluations in parallel.

    processes=None uses ProcessPoolExecutor's default worker count.

    Returns:
        runs: One row per model per split, with split seed, metrics,
              and confusion-matrix counts.
        summary: Mean precision, recall, and F1, plus summed
                 confusion-matrix counts for each model.
    """
    if iterations < 1:
        raise ValueError("iterations must be at least 1")
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")
    if not subsets:
        raise ValueError("subsets must contain at least one feature subset")
    if not neighbor_values or any(k < 1 for k in neighbor_values):
        raise ValueError("neighbor_values must contain positive integers")
    if processes is not None and processes < 1:
        raise ValueError("processes must be at least 1 or None")

    master_seed = (
        int(seed)
        if seed is not None
        else int(np.random.SeedSequence().generate_state(1)[0])
    )
    rng = np.random.default_rng(master_seed)
    split_seeds = rng.integers(
        0,
        np.iinfo(np.int32).max,
        size=iterations,
    )

    # Convert Cleveland's num values to binary: 0 versus greater than 0.
    target = pd.to_numeric(data[target_column], errors="coerce")
    valid_rows = target.notna()
    y = (target.loc[valid_rows] > 0).astype(int).to_numpy()
    data = data.loc[valid_rows]

    if set(np.unique(y)) != {0, 1}:
        raise ValueError(
            "The usable target data must contain both classes 0 and 1"
        )

    normalized_subsets = []
    for subset in subsets:
        subset = tuple(subset)

        if not subset:
            raise ValueError("Feature subsets cannot be empty")
        if len(set(subset)) != len(subset):
            raise ValueError(f"Subset contains duplicate columns: {subset}")
        if target_column in subset:
            raise ValueError("The target column cannot also be a feature")

        missing = set(subset) - set(data.columns)
        if missing:
            raise ValueError(
                f"Columns not found in data: {sorted(missing)}"
            )

        normalized_subsets.append(subset)

    row_ids = np.arange(len(data))
    split_indices = []

    # Generate each split once and reuse it for every model.
    for split_seed in split_seeds:
        train_ids, test_ids = train_test_split(
            row_ids,
            test_size=test_size,
            random_state=int(split_seed),
            stratify=y,
        )
        split_indices.append(
            (int(split_seed), train_ids, test_ids)
        )

    largest_train_set = min(
        len(train_ids)
        for _, train_ids, _ in split_indices
    )
    if any(k > largest_train_set for k in neighbor_values):
        raise ValueError(
            "Every n_neighbors value must be no larger than the "
            f"smallest training set ({largest_train_set} rows)"
        )

    tasks = [
        (subset, split_index)
        for subset in normalized_subsets
        for split_index in range(iterations)
    ]

    rows = []
    with ProcessPoolExecutor(
        max_workers=processes,
        initializer=_init_knn_worker,
        initargs=(
            data,
            y,
            split_indices,
            tuple(neighbor_values),
        ),
    ) as executor:
        for task_rows in executor.map(_evaluate_subset_split, tasks):
            rows.extend(task_rows)

    runs = pd.DataFrame(rows)
    runs.attrs["master_seed"] = master_seed

    summary = (
        runs.groupby(
            ["attributes", "n_neighbors"],
            as_index=False,
            sort=False,
        )
        .agg(
            mean_precision=("precision", "mean"),
            mean_recall=("recall", "mean"),
            mean_f1=("f1", "mean"),
            tn=("tn", "sum"),
            fp=("fp", "sum"),
            fn=("fn", "sum"),
            tp=("tp", "sum"),
        )
        .sort_values("mean_f1", ascending=False)
        .reset_index(drop=True)
    )

    return runs, summary

In [ ]:
# The target column is "num"; do not include it in the feature subsets.
feature_columns = [
    column for column in cleveland_data.columns
    if column != "num" and column != "heart_disease"
]

print(feature_columns)

subsets = list(combinations(feature_columns, 3))

runs, summary = run_knn_on_subsets(
    data=cleveland_data,
    subsets=subsets,
    target_column="heart_disease",
    test_size=0.2,
    seed=None,
    iterations=10,
    neighbor_values=(1, 3, 5, 7, 9, 11),
)

print("Master seed:", runs.attrs["master_seed"])
print(summary.head(20))

Part 2: Board Game Recommendation

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import requests
import xml.etree.ElementTree as ET
import time
import pandas as pd

# Replace 'your_file.csv' with the actual path or name of your CSV file
df = pd.read_csv('boardgames_ranks.csv')


def get_bgg_game_info_batch(game_ids):
    games_data = []
    # BGG API recommends batches of a few hundred to avoid timeouts
    batch_size = 300 
    
    for i in range(0, len(game_ids), batch_size):
        batch = game_ids[i:i + batch_size]
        ids_str = ",".join(map(str, batch))
        
        # BGG XML API2 endpoint
        url = f"https://boardgamegeek.com/xmlapi2/thing?id={ids_str}&stats=1"
        headers = {"User-Agent": "JupyterBGGDataFetcher/1.0"}
        
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            
            for item in root.findall('item'):
                def get_val(element):
                    return element.attrib.get('value') if element is not None else None
                
                ratings = item.find('statistics/ratings')
                
                games_data.append({
                    "id": int(item.attrib.get('id')),
                    "min_players": get_val(item.find('minplayers')),
                    "max_players": get_val(item.find('maxplayers')),
                    "playtime_mins": get_val(item.find('playingtime')),
                    "min_age": get_val(item.find('minage')),
                    "complexity_weight": get_val(ratings.find('averageweight') if ratings is not None else None),
                })
        else:
            print(f"Error fetching batch {i}-{i+batch_size}: HTTP {response.status_code}")
            
        # Politeness delay between batch requests
        time.sleep(2) 
        
    return pd.DataFrame(games_data)

# 1. Extract IDs from your existing baseGames DataFrame
ids_to_fetch = baseGames['id'].tolist()

# 2. Fetch the extra stats (Note: this might take a minute or two depending on dataset size)
print(f"Fetching BGG stats for {len(ids_to_fetch)} games in batches...")
bgg_stats_df = get_bgg_game_info_batch(ids_to_fetch)

# 3. Ensure both ID columns are integers for a clean merge
bgg_stats_df['id'] = bgg_stats_df['id'].astype(int)
baseGames['id'] = baseGames['id'].astype(int)

# 4. Merge the new stats into the baseGames DataFrame
extended_df = pd.merge(baseGames, bgg_stats_df, on='id', how='left')

# 5. Convert the new columns to numeric types (handling any missing values gracefully)
numeric_cols = ['min_players', 'max_players', 'playtime_mins', 'min_age', 'complexity_weight']
for col in numeric_cols:
    extended_df[col] = pd.to_numeric(extended_df[col], errors='coerce')

# Display the newly merged DataFrame
extended_df.head()
